# Fix `suggestion` = worst of the 3 horizons

The ML dataset was regenerated with the nested targets
(`future_deficit_1m/3m/6m`), but its `suggestion` column still uses the
**old** logic: it classifies only `future_deficit_6m`.

The agreed design is `suggestion` = the **most severe** class among the
three horizons (LOW < MEDIUM < HIGH < NOT_SUITABLE), so short but intense
water-stress peaks are not masked by the 6-month average.

This notebook:
1. Backs up the current CSV.
2. Recomputes `suggestion` using the same functions as the pipeline
   (`water_balance.classify_suggestion` / `worst_suggestion`).
3. Compares before/after and overwrites only that column.

No re-download is needed: the three target columns are already correct.

In [ ]:
import sys
from datetime import datetime
from pathlib import Path

import pandas as pd

# Make the analysis folder importable (where water_balance.py lives).
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "test" else Path.cwd()
ANALYSIS_DIR = PROJECT_ROOT / "analysis"
if str(ANALYSIS_DIR) not in sys.path:
    sys.path.insert(0, str(ANALYSIS_DIR))

import water_balance as wb

CSV_PATH = PROJECT_ROOT / "ml" / "ml_dataset_cacao_ccn51.csv"
print(f"Project root : {PROJECT_ROOT}")
print(f"Dataset path : {CSV_PATH}")
print(f"Exists       : {CSV_PATH.exists()}")

In [ ]:
# 1. Backup the current CSV (timestamped).
stamp = datetime.now().strftime("%y%m%d%H%M%S")
backup_path = CSV_PATH.with_name(f"ml_dataset_cacao_ccn51.BACKUP-v{stamp}.csv")

df = pd.read_csv(CSV_PATH)
df.to_csv(backup_path, index=False)
print(f"Backup saved to: {backup_path.name} ({len(df)} rows)")

In [ ]:
# 2. Show the current (old) suggestion distribution.
old_counts = df["suggestion"].value_counts()
print("BEFORE (old logic: 6m only):")
print(old_counts.to_string())

In [ ]:
# 3. Recompute suggestion = worst class among the three horizons.
targets = ["future_deficit_1m", "future_deficit_3m", "future_deficit_6m"]

def row_suggestion(row: pd.Series) -> str:
    classes = [wb.classify_suggestion(row[c]) for c in targets]
    return wb.worst_suggestion(*classes)

df["suggestion"] = df.apply(row_suggestion, axis=1)
print("Recomputed suggestion for all rows.")

In [ ]:
# 4. Compare before / after.
new_counts = df["suggestion"].value_counts()
compare = (
    pd.DataFrame({"before": old_counts, "after": new_counts})
    .reindex(["LOW", "MEDIUM", "HIGH", "NOT_SUITABLE"])
    .fillna(0)
    .astype(int)
)
compare["delta"] = compare["after"] - compare["before"]
print("AFTER (worst of 3) vs BEFORE (6m only):")
print(compare.to_string())

In [ ]:
# 5. Sanity checks before writing.
assert df["suggestion"].isna().sum() == 0, "unexpected NaN in suggestion"
assert set(df["suggestion"].unique()) <= {"LOW", "MEDIUM", "HIGH", "NOT_SUITABLE"}
assert len(df) == 26061, "row count changed unexpectedly"
print("Sanity checks passed.")

In [ ]:
# 6. Overwrite the dataset (only the suggestion column changed).
df.to_csv(CSV_PATH, index=False)
print(f"Dataset updated: {CSV_PATH.name}")
print(f"Rows={len(df)}, columns={len(df.columns)}")

## Result

`suggestion` now reflects the **most severe** of the three horizons. Short
but intense stress peaks (visible in `future_deficit_1m`) are no longer
masked by the 6-month average, so HIGH / NOT_SUITABLE counts increase while
LOW decreases.

The three target columns and all features are unchanged; only `suggestion`
was rewritten. A timestamped backup of the original CSV sits next to the
dataset for rollback.